# Kirchhoff & Bunsen (1860) — Chemical Analysis by Spectral Observations
# 분광 관측에 의한 화학 분석 — 구현

This notebook implements and visualizes the key concepts from Kirchhoff & Bunsen's 1860 paper:

이 노트북은 Kirchhoff & Bunsen의 1860년 논문의 핵심 개념을 구현하고 시각화합니다:

1. **Part 1**: Emission line spectra of alkali & alkaline earth metals / 알칼리 및 알칼리 토금속의 방출선 스펙트럼
2. **Part 2**: Flame temperature estimation / 불꽃 온도 추정
3. **Part 3**: Temperature invariance of spectral lines / 스펙트럼 선의 온도 불변성
4. **Part 4**: Matching emission lines to Fraunhofer absorption lines / 방출선과 Fraunhofer 흡수선의 대조
5. **Part 5**: Detection sensitivity comparison / 검출 민감도 비교
6. **Part 6**: Simulating a mixture analysis / 혼합 시료 분석 시뮬레이션
7. **Part 7**: Spectral fingerprinting — element identification / 스펙트럼 지문 — 원소 식별
8. **Part 8**: Connection to modern spectroscopy / 현대 분광학과의 연결

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgba
from matplotlib import colormaps

plt.rcParams.update({
    'figure.figsize': (14, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

In [ ]:
def wavelength_to_rgb(wavelength_nm):
    """Convert a visible wavelength (380-750 nm) to an approximate RGB color.

    Args:
        wavelength_nm: Wavelength in nanometers.

    Returns:
        Tuple of (R, G, B) values in [0, 1].
    """
    w = wavelength_nm
    if w < 380 or w > 750:
        return (0.5, 0.5, 0.5)

    # Peicewise linear approximation of spectral colors.
    if 380 <= w < 440:
        r = -(w - 440) / (440 - 380)
        g = 0.0
        b = 1.0
    elif 440 <= w < 490:
        r = 0.0
        g = (w - 440) / (490 - 440)
        b = 1.0
    elif 490 <= w < 510:
        r = 0.0
        g = 1.0
        b = -(w - 510) / (510 - 490)
    elif 510 <= w < 580:
        r = (w - 510) / (580 - 510)
        g = 1.0
        b = 0.0
    elif 580 <= w < 645:
        r = 1.0
        g = -(w - 645) / (645 - 580)
        b = 0.0
    else:
        r = 1.0
        g = 0.0
        b = 0.0

    # Intensity factor for edges of visible range.
    if 380 <= w < 420:
        factor = 0.3 + 0.7 * (w - 380) / (420 - 380)
    elif 700 < w <= 750:
        factor = 0.3 + 0.7 * (750 - w) / (750 - 700)
    else:
        factor = 1.0

    return (r * factor, g * factor, b * factor)


def draw_spectrum_background(ax, wl_min=380, wl_max=750, y_min=0, y_max=1):
    """Draw a continuous visible spectrum as background on the given axes.

    Args:
        ax: Matplotlib axes object.
        wl_min: Minimum wavelength in nm.
        wl_max: Maximum wavelength in nm.
        y_min: Bottom of the spectrum band.
        y_max: Top of the spectrum band.
    """
    n_steps = 500
    wavelengths = np.linspace(wl_min, wl_max, n_steps)
    for i in range(len(wavelengths) - 1):
        color = wavelength_to_rgb(wavelengths[i])
        ax.axvspan(wavelengths[i], wavelengths[i + 1],
                   ymin=y_min, ymax=y_max, color=color, alpha=0.4)

print("Utility functions loaded.")

## Part 1: Emission Line Spectra of Elements / 원소별 방출선 스펙트럼

Kirchhoff & Bunsen은 6개 원소(Na, Li, K, Sr, Ca, Ba) 각각의 고유한 방출선 스펙트럼을 체계적으로 기록했습니다. 각 원소는 고유한 파장 위치에 밝은 선을 보이며, 이것이 원소의 "스펙트럼 지문"입니다.

Each element produces emission lines at unique, fixed wavelengths — its spectral "fingerprint."

$$\text{Element} \xrightarrow{\text{heat}} \text{Excited atoms} \xrightarrow{\text{de-excitation}} \text{Photons at specific } \lambda$$

In [ ]:
# Spectral line data for the 6 elements studied by Kirchhoff & Bunsen.
# Wavelengths in nm, with relative intensities (arbitrary scale 0-1).
# Using Kirchhoff & Bunsen's own Greek letter notation.

ELEMENTS = {
    'Na (Natrium)': {
        'lines': {
            r'Na $\alpha$ (D)': (589.0, 1.0),   # The famous D line (doublet)
            r'Na $\alpha_2$ (D₂)': (589.6, 0.9),
        },
        'color': '#FFD700',
    },
    'Li (Lithium)': {
        'lines': {
            r'Li $\alpha$': (670.8, 1.0),   # Intense red
            r'Li $\beta$': (610.4, 0.2),    # Faint yellow
        },
        'color': '#FF3333',
    },
    'K (Kalium)': {
        'lines': {
            r'Ka $\alpha$ (≈A)': (766.5, 0.8),  # Far red, near Fraunhofer A
            r'Ka $\beta$': (404.4, 0.5),          # Violet
        },
        'color': '#9933FF',
    },
    'Sr (Strontium)': {
        'lines': {
            r'Sr $\alpha$': (605.0, 0.7),   # Orange
            r'Sr $\beta$': (687.0, 0.8),    # Red
            r'Sr $\gamma$': (707.0, 0.6),   # Red
            r'Sr $\delta$': (460.7, 0.9),   # Blue — key diagnostic line
        },
        'color': '#FF6633',
    },
    'Ca (Calcium)': {
        'lines': {
            r'Ca $\alpha$': (622.0, 0.7),   # Orange
            r'Ca $\beta$': (518.9, 1.0),    # Green — most distinctive
            r'Ca H': (396.8, 0.6),          # Violet (Ca II H)
            r'Ca K': (393.4, 0.7),          # Violet (Ca II K)
        },
        'color': '#33CC66',
    },
    'Ba (Baryum)': {
        'lines': {
            r'Ba $\alpha$': (553.5, 1.0),   # Green
            r'Ba $\beta$': (493.4, 0.9),    # Blue-green
            r'Ba $\gamma$': (649.7, 0.4),   # Red
        },
        'color': '#33CC33',
    },
}

# Plot emission spectra for all 6 elements.
fig, axes = plt.subplots(6, 1, figsize=(14, 12), sharex=True)
fig.suptitle("Emission Line Spectra of 6 Elements (Kirchhoff & Bunsen 1860)\n"
             "6개 원소의 방출선 스펙트럼", fontsize=16, y=1.02)

for idx, (element, data) in enumerate(ELEMENTS.items()):
    ax = axes[idx]
    draw_spectrum_background(ax, wl_min=380, wl_max=780)

    for label, (wl, intensity) in data['lines'].items():
        color = wavelength_to_rgb(wl)
        # Draw bright emission line.
        ax.axvline(wl, color=color, linewidth=2.5 * intensity + 0.5,
                   alpha=0.9, zorder=5)
        # Add glow effect.
        ax.axvline(wl, color=color, linewidth=8 * intensity,
                   alpha=0.25, zorder=4)
        ax.text(wl, 0.85, label, fontsize=7, ha='center', va='top',
                rotation=90, transform=ax.get_xaxis_transform(),
                color='white', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.1', fc='black', alpha=0.6))

    ax.set_ylabel(element, fontsize=11, fontweight='bold',
                  color=data['color'], rotation=0, labelpad=90, va='center')
    ax.set_yticks([])
    ax.set_xlim(380, 780)
    ax.set_facecolor('black')

axes[-1].set_xlabel("Wavelength / 파장 (nm)", fontsize=13)
plt.tight_layout()
plt.show()

## Part 2: Flame Temperature Estimation / 불꽃 온도 추정

Kirchhoff & Bunsen은 에너지 보존 법칙을 이용하여 다양한 불꽃의 온도를 추정했습니다:

$$t = \frac{\sum g_i \cdot w_i}{\sum p_j \cdot s_j}$$

여기서 $g$는 연소물 질량, $w$는 연소열, $p$는 생성물 질량, $s$는 비열입니다. 이것은 연소의 총 발열량을 생성물의 열용량으로 나눈 것으로, 단열 화염 온도(adiabatic flame temperature)의 초기 형태입니다.

In [ ]:
def estimate_flame_temperature(combustion_heats, specific_heats):
    """Estimate flame temperature using Kirchhoff & Bunsen's formula.

    Args:
        combustion_heats: List of (mass_g, heat_of_combustion_cal_per_g) tuples.
        specific_heats: List of (mass_g, specific_heat_cal_per_g_C) tuples.

    Returns:
        Estimated temperature in degrees Celsius.
    """
    numerator = sum(g * w for g, w in combustion_heats)
    denominator = sum(p * s for p, s in specific_heats)
    return numerator / denominator


# Data from Kirchhoff & Bunsen's paper (Table on pp. 164-165).
# Combustion heat values in degrees Celsius (their units).
flames = {
    'Schwefelflamme\n(Sulfur)': {
        'measured_T': 1820,
        'description': 'S + O₂ → SO₂',
    },
    'CS₂-flamme\n(Carbon disulfide)': {
        'measured_T': 2195,
        'description': 'CS₂ + 3O₂ → CO₂ + 2SO₂',
    },
    'Leuchtgasflamme\n(Illuminating gas)': {
        'measured_T': 2350,
        'description': 'CH₄ + 2O₂ → CO₂ + 2H₂O',
    },
    'CO-flamme\n(Carbon monoxide)': {
        'measured_T': 3042,
        'description': '2CO + O₂ → 2CO₂',
    },
    'H₂ in Luft\n(Hydrogen in air)': {
        'measured_T': 3259,
        'description': '2H₂ + O₂ → 2H₂O',
    },
    'Knallgasflamme\n(Oxyhydrogen)': {
        'measured_T': 8061,
        'description': '2H₂ + O₂ → 2H₂O (pure O₂)',
    },
}

# Visualization.
fig, ax = plt.subplots(figsize=(12, 7))

names = list(flames.keys())
temps = [flames[n]['measured_T'] for n in names]
descriptions = [flames[n]['description'] for n in names]

# Color gradient from cool to hot.
colors = plt.cm.YlOrRd(np.linspace(0.2, 0.95, len(names)))

bars = ax.barh(range(len(names)), temps, color=colors, edgecolor='black',
               linewidth=0.5, height=0.6)

for i, (bar, temp, desc) in enumerate(zip(bars, temps, descriptions)):
    ax.text(temp + 100, i, f'{temp}°C', va='center', fontsize=12,
            fontweight='bold')
    ax.text(temp / 2, i, desc, va='center', ha='center', fontsize=9,
            color='black', style='italic')

ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=10)
ax.set_xlabel("Estimated Temperature / 추정 온도 (°C)", fontsize=13)
ax.set_title("Flame Temperatures Estimated by Kirchhoff & Bunsen (1860)\n"
             "Kirchhoff & Bunsen이 추정한 불꽃 온도 — "
             r"$t = \sum gw / \sum ps$",
             fontsize=14)
ax.set_xlim(0, 9500)

# Highlight key finding: spectral lines unchanged across this range!
ax.axvline(1820, color='blue', linestyle='--', alpha=0.3)
ax.axvline(8061, color='red', linestyle='--', alpha=0.3)
ax.annotate('', xy=(8061, 6.5), xytext=(1820, 6.5),
            arrowprops=dict(arrowstyle='<->', color='darkred', lw=2))
ax.text(4940, 6.7, '4.4× temperature range\nyet spectral lines UNCHANGED!',
        ha='center', fontsize=11, color='darkred', fontweight='bold')

ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Part 3: Temperature Invariance of Spectral Lines / 스펙트럼 선의 온도 불변성

Kirchhoff & Bunsen의 가장 중요한 실험적 발견: 온도가 변하면 선의 **강도(intensity)**는 변하지만, **위치(wavelength)**는 변하지 않습니다. 이것을 시뮬레이션으로 보여줍니다.

현대 물리학의 관점에서, 이것은 원자의 전자 에너지 준위가 외부 온도에 의존하지 않기 때문입니다:

$$E_{\text{photon}} = h\nu = \frac{hc}{\lambda} = E_n - E_m$$

에너지 준위 $E_n$과 $E_m$은 원자 고유의 성질이므로, 방출되는 광자의 파장 $\lambda$도 원자 고유입니다.

In [ ]:
# Simulate Na emission spectrum at different flame temperatures.
# Key point: line POSITION stays fixed, only INTENSITY changes.

na_lines_nm = [589.0, 589.6]  # Na D doublet.
flame_temps = [1820, 2350, 3259, 8061]
flame_names = ['Sulfur (1820°C)', 'Leuchtgas (2350°C)',
               'H₂ in air (3259°C)', 'Oxyhydrogen (8061°C)']

wavelengths = np.linspace(570, 610, 2000)


def emission_spectrum(wavelengths, line_centers, temperature, sigma=0.3):
    """Simulate emission spectrum as sum of Gaussians.

    Intensity scales with temperature (simplified Boltzmann-like).

    Args:
        wavelengths: Array of wavelengths in nm.
        line_centers: List of line center wavelengths.
        temperature: Flame temperature in Celsius.
        sigma: Line width in nm (Gaussian sigma).

    Returns:
        Array of intensities.
    """
    intensity_scale = (temperature / 8061) ** 1.5  # Approximate scaling.
    spectrum = np.zeros_like(wavelengths)
    for center in line_centers:
        spectrum += np.exp(-0.5 * ((wavelengths - center) / sigma) ** 2)
    return spectrum * intensity_scale


fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
fig.suptitle("Na D Line at Different Flame Temperatures\n"
             "다른 불꽃 온도에서의 Na D선 — "
             "Position unchanged, intensity varies",
             fontsize=15, y=1.02)

for idx, (temp, name) in enumerate(zip(flame_temps, flame_names)):
    ax = axes[idx]
    spectrum = emission_spectrum(wavelengths, na_lines_nm, temp)
    color_intensity = 0.3 + 0.7 * (temp / 8061)
    ax.fill_between(wavelengths, spectrum, alpha=0.7,
                    color=(1.0, color_intensity * 0.8, 0.0))
    ax.plot(wavelengths, spectrum, color='darkorange', linewidth=1.5)

    # Mark exact line positions with dashed lines.
    for lc in na_lines_nm:
        ax.axvline(lc, color='white', linestyle='--', alpha=0.5, linewidth=0.8)

    ax.set_ylabel(name, fontsize=10, rotation=0, labelpad=120, va='center')
    ax.set_ylim(0, 1.15)
    ax.set_yticks([])
    ax.set_facecolor('#1a1a2e')

    # Show temperature and peak intensity.
    ax.text(572, 0.8, f'T = {temp}°C\nPeak = {spectrum.max():.2f}',
            fontsize=10, color='white',
            bbox=dict(boxstyle='round', fc='black', alpha=0.5))

# Mark constant position.
for lc in na_lines_nm:
    axes[0].annotate(f'{lc} nm', xy=(lc, 1.1),
                     fontsize=10, ha='center', color='cyan', fontweight='bold')

axes[-1].set_xlabel("Wavelength / 파장 (nm)", fontsize=13)
plt.tight_layout()
plt.show()

print("핵심 관찰: 온도가 1820°C → 8061°C로 4.4배 변해도,")
print("Na D선의 위치(589.0 nm, 589.6 nm)는 정확히 동일합니다.")
print("변하는 것은 오직 강도(intensity)뿐입니다.")

## Part 4: Emission Lines = Fraunhofer Absorption Lines / 방출선 = Fraunhofer 흡수선 대조

이것이 태양 물리학에서 가장 중요한 연결입니다. 불꽃의 밝은 방출선과 태양의 어두운 Fraunhofer 선이 **정확히 같은 파장**에 위치합니다.

Kirchhoff의 복사 법칙에 의한 해석:
- 뜨거운 태양 내부(광구) → **연속 스펙트럼** 방출
- 더 차가운 태양 대기 → 연속 스펙트럼의 특정 파장을 **흡수** → 어두운 선
- 같은 원소의 불꽃 → 같은 파장에서 **방출** → 밝은 선

In [ ]:
# Compare emission lines (flame) vs. absorption lines (solar / Fraunhofer).
# Demonstrate the fundamental connection discovered by Kirchhoff.

fraunhofer_lines = {
    'A': (759.4, 'O₂ (telluric)', 0.7),
    'B': (686.7, 'O₂ (telluric)', 0.8),
    'C': (656.3, 'H (Hα)', 0.6),
    'D': (589.3, 'Na', 1.0),
    'E': (527.0, 'Fe', 0.5),
    'F': (486.1, 'H (Hβ)', 0.7),
    'G': (430.8, 'Fe, Ca', 0.6),
    'H': (396.8, 'Ca⁺', 0.8),
    'K': (393.4, 'Ca⁺', 0.9),
}

# Matching pairs: flame emission line → Fraunhofer absorption line.
matches = {
    r'Na $\alpha$': {'emission': 589.0, 'fraunhofer': 'D', 'fraunhofer_wl': 589.3},
    r'Ka $\alpha$': {'emission': 766.5, 'fraunhofer': 'A', 'fraunhofer_wl': 759.4},
    r'Ca H': {'emission': 396.8, 'fraunhofer': 'H', 'fraunhofer_wl': 396.8},
    r'Ca K': {'emission': 393.4, 'fraunhofer': 'K', 'fraunhofer_wl': 393.4},
}

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
fig.suptitle("Emission Lines (Flame) vs. Absorption Lines (Sun / Fraunhofer)\n"
             "방출선 (불꽃) vs. 흡수선 (태양 / Fraunhofer)",
             fontsize=15, y=1.02)

wl_range = np.linspace(380, 780, 3000)

# --- Panel 1: Solar (absorption) spectrum ---
ax1.set_title("Solar Spectrum with Fraunhofer Lines (Absorption) / "
              "태양 스펙트럼 — Fraunhofer 흡수선", fontsize=12)
draw_spectrum_background(ax1, 380, 780)

# Draw Fraunhofer dark lines.
for letter, (wl, elem, strength) in fraunhofer_lines.items():
    ax1.axvline(wl, color='black', linewidth=2.5 * strength,
                alpha=0.8, zorder=5)
    ax1.text(wl, 1.05, f'{letter}\n({elem})', fontsize=7, ha='center',
             va='bottom', fontweight='bold',
             transform=ax1.get_xaxis_transform())

ax1.set_ylabel("Solar\n태양", fontsize=11, rotation=0, labelpad=50)
ax1.set_yticks([])
ax1.set_facecolor('#FFF8DC')

# --- Panel 2: Flame emission spectrum (all 6 elements combined) ---
ax2.set_title("Flame Emission Lines (Laboratory) / "
              "불꽃 방출선 (실험실)", fontsize=12)

for element, data in ELEMENTS.items():
    for label, (wl, intensity) in data['lines'].items():
        color = wavelength_to_rgb(wl)
        ax2.axvline(wl, color=color, linewidth=3 * intensity,
                    alpha=0.9, zorder=5)
        ax2.axvline(wl, color=color, linewidth=10 * intensity,
                    alpha=0.2, zorder=4)

ax2.set_ylabel("Flame\n불꽃", fontsize=11, rotation=0, labelpad=50)
ax2.set_yticks([])
ax2.set_facecolor('black')

# --- Panel 3: Matching diagram ---
ax3.set_title("Wavelength Matches: Emission = Absorption / "
              "파장 일치: 방출 = 흡수", fontsize=12)

for i, (label, match) in enumerate(matches.items()):
    em_wl = match['emission']
    fr_wl = match['fraunhofer_wl']
    fr_letter = match['fraunhofer']
    color = wavelength_to_rgb(em_wl)

    y_pos = 0.2 + i * 0.2
    ax3.plot([em_wl, fr_wl], [y_pos - 0.05, y_pos + 0.05],
             'o-', color=color, markersize=8, linewidth=2)
    ax3.text(em_wl + 15, y_pos,
             f'{label} ({em_wl} nm) ↔ Fraunhofer {fr_letter} ({fr_wl} nm)',
             fontsize=9, va='center', fontweight='bold', color=color)

ax3.set_ylabel("Match\n대조", fontsize=11, rotation=0, labelpad=50)
ax3.set_yticks([])
ax3.set_facecolor('#1a1a2e')
ax3.set_xlim(380, 780)

axes_list = [ax1, ax2, ax3]
for ax in axes_list:
    ax.set_xlim(380, 780)

ax3.set_xlabel("Wavelength / 파장 (nm)", fontsize=13)
plt.tight_layout()
plt.show()

print("핵심 발견: 불꽃의 밝은 방출선 = 태양의 어두운 Fraunhofer 흡수선")
print("→ 태양 대기에 Na, K, Ca 등이 존재한다는 증거!")

## Part 5: Detection Sensitivity Comparison / 검출 민감도 비교

Kirchhoff & Bunsen은 분광법이 기존 습식 화학 분석보다 수십만 배 민감하다고 보고했습니다. 특히 나트륨의 경우 **1/3,000,000 mg** (~0.3 피코그램)까지 검출 가능했습니다.

In [ ]:
# Detection limits reported by Kirchhoff & Bunsen (in mg).
detection_limits = {
    'Na': 1 / 3_000_000,     # 0.33 pg
    'Li': 9 / 1000,          # 9 μg (per 60m³ air)
    'K': 1 / 1000,           # 1 μg
    'Ca': 6 / 1_000_000,     # 6 ng
    'Sr': 6 / 100_000,       # 60 ng
    'Ba': 1 / 1000,          # 1 μg
}

# Approximate wet chemistry detection limits for comparison (ca. 1860).
wet_chem_limits = {
    'Na': 0.1,     # ~0.1 mg
    'Li': 1.0,     # ~1 mg
    'K': 0.5,      # ~0.5 mg
    'Ca': 0.05,    # ~0.05 mg
    'Sr': 0.1,     # ~0.1 mg
    'Ba': 0.5,     # ~0.5 mg
}

fig, ax = plt.subplots(figsize=(14, 7))

elements = list(detection_limits.keys())
spectral_vals = [detection_limits[e] for e in elements]
wet_vals = [wet_chem_limits[e] for e in elements]
improvement = [w / s for w, s in zip(wet_vals, spectral_vals)]

x = np.arange(len(elements))
width = 0.35

bars1 = ax.bar(x - width / 2, wet_vals, width, label='Wet Chemistry (~1860)',
               color='#8B4513', edgecolor='black', alpha=0.8)
bars2 = ax.bar(x + width / 2, spectral_vals, width,
               label='Spectral Analysis (K&B)',
               color='#FFD700', edgecolor='black', alpha=0.8)

ax.set_yscale('log')
ax.set_ylabel("Detection Limit / 검출 한계 (mg)", fontsize=13)
ax.set_xlabel("Element / 원소", fontsize=13)
ax.set_title("Detection Sensitivity: Spectral Analysis vs. Wet Chemistry\n"
             "검출 민감도: 분광 분석 vs. 습식 화학 분석",
             fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(elements, fontsize=12, fontweight='bold')
ax.legend(fontsize=11)

# Annotate improvement factors.
for i, (elem, imp) in enumerate(zip(elements, improvement)):
    y_pos = max(spectral_vals[i], wet_vals[i]) * 2
    ax.text(i, y_pos, f'{imp:,.0f}×\nbetter',
            ha='center', fontsize=10, fontweight='bold', color='darkred')

ax.set_ylim(1e-8, 10)
ax.grid(axis='y', alpha=0.3)

# Add scale annotations.
ax.axhline(1e-3, color='gray', linestyle=':', alpha=0.5)
ax.text(5.5, 1.5e-3, '1 μg', fontsize=9, color='gray')
ax.axhline(1e-6, color='gray', linestyle=':', alpha=0.5)
ax.text(5.5, 1.5e-6, '1 ng', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

print(f"\nNa 민감도 비교:")
print(f"  습식 화학: ~0.1 mg = 100 μg")
print(f"  분광 분석: {1/3_000_000:.2e} mg = ~0.33 pg")
print(f"  향상 배율: {0.1 / (1/3_000_000):,.0f}배")

## Part 6: Simulating a Mixture Analysis / 혼합 시료 분석 시뮬레이션

Kirchhoff & Bunsen은 NaCl, KCl, LiCl, CaCl₂, SrCl₂, BaCl₂의 6종 혼합물을 불꽃에서 분석하는 과정을 상세히 기술했습니다. 각 원소의 휘발성 차이에 의해 시간순으로 다른 원소가 나타납니다:

1. **Na** (가장 먼저, 가장 강렬) → 2. **Li, K** → 3. **Ba** → 4. **Ca, Sr** (마지막, 가장 오래 지속)

이 시간 순서를 시뮬레이션합니다.

In [ ]:
# Simulate the time evolution of a 6-element mixture in a flame.
# Each element appears and fades at different rates based on volatility.

# Time profile parameters: (onset_s, peak_s, fade_s) — approximate.
time_profiles = {
    'Na': {'onset': 0, 'peak': 1, 'fade': 8, 'max_intensity': 1.0},
    'Li': {'onset': 2, 'peak': 4, 'fade': 10, 'max_intensity': 0.8},
    'K':  {'onset': 2, 'peak': 5, 'fade': 12, 'max_intensity': 0.5},
    'Ba': {'onset': 5, 'peak': 10, 'fade': 25, 'max_intensity': 0.6},
    'Ca': {'onset': 8, 'peak': 20, 'fade': 60, 'max_intensity': 0.7},
    'Sr': {'onset': 10, 'peak': 25, 'fade': 70, 'max_intensity': 0.65},
}

element_colors = {
    'Na': '#FFD700', 'Li': '#FF3333', 'K': '#9933FF',
    'Ba': '#33CC33', 'Ca': '#33CC66', 'Sr': '#FF6633',
}

# Primary emission line for each element.
primary_lines = {
    'Na': [589.0], 'Li': [670.8], 'K': [766.5],
    'Ba': [553.5, 493.4], 'Ca': [518.9, 622.0], 'Sr': [460.7, 605.0],
}

time = np.linspace(0, 80, 500)


def time_envelope(t, onset, peak, fade, max_intensity):
    """Compute intensity envelope over time for a volatilizing element.

    Args:
        t: Time array in seconds.
        onset: Time when element starts appearing.
        peak: Time of peak intensity.
        fade: Time when element has mostly faded.
        max_intensity: Maximum intensity at peak.

    Returns:
        Array of intensity values.
    """
    envelope = np.zeros_like(t)
    # Rising phase.
    mask_rise = (t >= onset) & (t < peak)
    envelope[mask_rise] = max_intensity * (
        (t[mask_rise] - onset) / (peak - onset))
    # Decay phase (exponential).
    mask_decay = t >= peak
    tau = (fade - peak) / 3  # e-folding time.
    envelope[mask_decay] = max_intensity * np.exp(
        -(t[mask_decay] - peak) / tau)
    return envelope


# --- Plot 1: Time evolution of each element's intensity ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10),
                                gridspec_kw={'height_ratios': [1, 2]})

fig.suptitle("6-Element Mixture Analysis in Flame (Kirchhoff & Bunsen)\n"
             "불꽃에서의 6종 혼합 시료 분석 — 시간 순서",
             fontsize=15, y=1.02)

for elem, params in time_profiles.items():
    envelope = time_envelope(time, **params)
    ax1.plot(time, envelope, color=element_colors[elem],
             linewidth=2.5, label=elem)
    ax1.fill_between(time, envelope, alpha=0.15,
                     color=element_colors[elem])

ax1.set_ylabel("Intensity / 강도", fontsize=12)
ax1.set_xlabel("Time / 시간 (seconds)", fontsize=12)
ax1.legend(ncol=6, loc='upper right', fontsize=10)
ax1.set_xlim(0, 80)
ax1.set_ylim(0, 1.1)
ax1.set_title("Temporal Evolution of Each Element / 각 원소의 시간적 변화",
              fontsize=12)

# Phase annotations.
ax1.axvspan(0, 5, alpha=0.05, color='yellow')
ax1.axvspan(5, 15, alpha=0.05, color='green')
ax1.axvspan(15, 80, alpha=0.05, color='blue')
ax1.text(2.5, 1.0, 'Phase 1\nNa, Li, K', ha='center', fontsize=8)
ax1.text(10, 1.0, 'Phase 2\nBa appears', ha='center', fontsize=8)
ax1.text(40, 1.0, 'Phase 3\nCa, Sr persist', ha='center', fontsize=8)

# --- Plot 2: Spectral snapshots at 4 time points ---
snapshot_times = [2, 8, 20, 50]
snapshot_labels = ['t=2s: Na dominant',
                   't=8s: Li, K, Ba emerging',
                   't=20s: Ca, Sr appearing',
                   't=50s: Ca, Sr persist']

wavelengths_full = np.linspace(380, 780, 3000)

for idx, (t_snap, label) in enumerate(zip(snapshot_times, snapshot_labels)):
    y_offset = (3 - idx) * 1.2

    # Compute spectrum at this time.
    spectrum = np.zeros_like(wavelengths_full)
    for elem, params in time_profiles.items():
        intensity_at_t = time_envelope(
            np.array([t_snap]), **params)[0]
        if intensity_at_t > 0.01:
            for line_wl in primary_lines[elem]:
                spectrum += intensity_at_t * np.exp(
                    -0.5 * ((wavelengths_full - line_wl) / 0.8) ** 2)

    ax2.fill_between(wavelengths_full, y_offset, y_offset + spectrum * 0.8,
                     alpha=0.6, color='orange')
    ax2.plot(wavelengths_full, y_offset + spectrum * 0.8,
             color='darkorange', linewidth=0.8)
    ax2.axhline(y_offset, color='gray', linewidth=0.3, alpha=0.5)
    ax2.text(385, y_offset + 0.5, label, fontsize=9, fontweight='bold',
             color='white',
             bbox=dict(boxstyle='round', fc='black', alpha=0.6))

ax2.set_facecolor('black')
ax2.set_xlim(380, 780)
ax2.set_yticks([])
ax2.set_xlabel("Wavelength / 파장 (nm)", fontsize=12)
ax2.set_title("Spectral Snapshots Over Time / 시간에 따른 스펙트럼 스냅샷",
              fontsize=12)

plt.tight_layout()
plt.show()

## Part 7: Spectral Fingerprinting — Element Identification / 스펙트럼 지문 — 원소 식별

분광 분석의 핵심 응용: 미지 시료의 스펙트럼을 관찰하고, 알려진 원소의 스펙트럼 데이터베이스와 대조하여 시료에 포함된 원소를 식별합니다.

이것을 구현하여, "미지 시료"의 스펙트럼에서 원소를 자동 식별하는 간단한 알고리즘을 만들어봅니다.

In [ ]:
# Build a simple spectral fingerprint database and identification system.
# This is the algorithmic essence of Kirchhoff & Bunsen's method.

# Database: known emission lines for each element (nm).
SPECTRAL_DB = {
    'Na': [589.0, 589.6],
    'Li': [670.8, 610.4],
    'K':  [766.5, 404.4],
    'Ca': [393.4, 396.8, 518.9, 622.0],
    'Sr': [460.7, 605.0, 687.0, 707.0],
    'Ba': [493.4, 553.5, 649.7],
    'Cs': [455.5, 459.3],  # Caesium — discovered by K&B!
}


def identify_elements(observed_lines, database, tolerance_nm=2.0):
    """Identify elements from observed spectral lines by matching.

    Args:
        observed_lines: List of observed wavelengths in nm.
        database: Dict mapping element names to lists of known lines.
        tolerance_nm: Maximum wavelength difference for a match.

    Returns:
        Dict mapping element names to lists of matched lines.
    """
    results = {}
    for element, known_lines in database.items():
        matched = []
        for obs in observed_lines:
            for known in known_lines:
                if abs(obs - known) <= tolerance_nm:
                    matched.append((obs, known))
                    break
        if matched:
            confidence = len(matched) / len(known_lines)
            results[element] = {
                'matches': matched,
                'confidence': confidence,
                'n_matched': len(matched),
                'n_total': len(known_lines),
            }
    return results


# Simulate "unknown" sample: seawater (Kirchhoff & Bunsen's example #1).
# Contains Na, Ca, K, Li, Sr traces.
unknown_sample_lines = [
    393.5,  # Ca K
    396.9,  # Ca H
    460.5,  # Sr δ
    519.0,  # Ca β
    589.1,  # Na α
    589.7,  # Na α₂
    605.2,  # Sr α
    622.1,  # Ca α
    670.9,  # Li α
    766.3,  # K α
]

results = identify_elements(unknown_sample_lines, SPECTRAL_DB)

# Visualize.
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8),
                                gridspec_kw={'height_ratios': [1, 1.5]})

fig.suptitle("Spectral Fingerprinting: Identifying Elements in Seawater\n"
             "스펙트럼 지문 분석: 해수 시료의 원소 식별",
             fontsize=15, y=1.02)

# Panel 1: Observed spectrum.
ax1.set_title("Observed Spectrum (Unknown Sample) / "
              "관측된 스펙트럼 (미지 시료)", fontsize=12)
ax1.set_facecolor('black')
for wl in unknown_sample_lines:
    color = wavelength_to_rgb(wl)
    ax1.axvline(wl, color=color, linewidth=3, alpha=0.9)
    ax1.axvline(wl, color=color, linewidth=10, alpha=0.2)
    ax1.text(wl, 0.95, f'{wl}', fontsize=7, ha='center',
             rotation=90, color='white',
             transform=ax1.get_xaxis_transform())
ax1.set_xlim(380, 780)
ax1.set_yticks([])
ax1.set_ylabel("Observed\n관측", rotation=0, labelpad=60, fontsize=11)

# Panel 2: Identification results.
ax2.set_title("Identification Results / 식별 결과", fontsize=12)

sorted_results = sorted(results.items(),
                        key=lambda x: x[1]['confidence'], reverse=True)

y_positions = np.arange(len(sorted_results))
confidences = [r[1]['confidence'] for r in sorted_results]
names = [f"{r[0]} ({r[1]['n_matched']}/{r[1]['n_total']} lines)"
         for r in sorted_results]

elem_bar_colors = {
    'Na': '#FFD700', 'Li': '#FF3333', 'K': '#9933FF',
    'Ca': '#33CC66', 'Sr': '#FF6633', 'Ba': '#33CC33', 'Cs': '#3366FF',
}
bar_colors = [elem_bar_colors.get(r[0], 'gray') for r in sorted_results]

bars = ax2.barh(y_positions, confidences, color=bar_colors,
                edgecolor='black', height=0.6, alpha=0.8)

for i, (name, conf) in enumerate(zip(names, confidences)):
    ax2.text(conf + 0.02, i, f'{conf:.0%}', va='center',
             fontsize=11, fontweight='bold')

ax2.set_yticks(y_positions)
ax2.set_yticklabels(names, fontsize=11)
ax2.set_xlabel("Confidence / 신뢰도", fontsize=12)
ax2.set_xlim(0, 1.15)
ax2.invert_yaxis()
ax2.axvline(0.5, color='red', linestyle='--', alpha=0.3)
ax2.text(0.52, len(sorted_results) - 0.5, 'threshold',
         fontsize=9, color='red', alpha=0.5)

plt.tight_layout()
plt.show()

print("\n식별 결과 — 해수(Meerwasser) 시료:")
for elem, info in sorted_results:
    status = "DETECTED" if info['confidence'] >= 0.5 else "trace"
    print(f"  {elem}: {info['n_matched']}/{info['n_total']} lines matched "
          f"({info['confidence']:.0%}) → {status}")

## Part 8: Connection to Modern Spectroscopy / 현대 분광학과의 연결

Kirchhoff & Bunsen의 1860년 방법은 현대 분석 화학의 직접적 조상입니다:

| 1860 (Kirchhoff & Bunsen) | 현대 동등 기법 | 개선점 |
|---|---|---|
| Flame emission spectroscopy | **AAS** (Atomic Absorption Spectroscopy) | 정량적 농도 측정 |
| Visual observation through eyepiece | **CCD/CMOS detectors** | 디지털 기록, 동시 전 파장 관측 |
| CS₂ prism, 60° | **Echelle diffraction grating** | 분해능 $R > 100,000$ |
| Wavelength matching by eye | **Automated line identification** | 데이터베이스 > 100만 선 |
| Sensitivity: ~pg | **ICP-MS** (Inductively Coupled Plasma Mass Spec.) | 감도: fg (펨토그램) |

Kirchhoff & Bunsen의 핵심 원리 — "각 원소는 고유한 스펙트럼 지문을 가진다" — 는 160년이 지난 현재에도 **변하지 않는 과학적 사실**입니다.

In [ ]:
# Timeline of spectroscopic discoveries enabled by Kirchhoff & Bunsen's method.

discoveries = [
    (1860, 'Cs', 'Caesium', 'Kirchhoff & Bunsen', '#3366FF'),
    (1861, 'Rb', 'Rubidium', 'Kirchhoff & Bunsen', '#CC0000'),
    (1861, 'Tl', 'Thallium', 'Crookes', '#33CC33'),
    (1863, 'In', 'Indium', 'Reich & Richter', '#6633CC'),
    (1868, 'He', 'Helium (in Sun!)', 'Janssen & Lockyer', '#FFCC00'),
    (1875, 'Ga', 'Gallium', 'Lecoq de Boisbaudran', '#CC6633'),
    (1879, 'Sm', 'Samarium', 'Lecoq de Boisbaudran', '#9966CC'),
    (1885, 'Pr', 'Praseodymium', 'von Welsbach', '#66CC99'),
    (1885, 'Nd', 'Neodymium', 'von Welsbach', '#6699CC'),
]

fig, ax = plt.subplots(figsize=(14, 6))

for i, (year, symbol, name, discoverer, color) in enumerate(discoveries):
    ax.barh(i, year - 1859, left=1859, color=color, height=0.6,
            edgecolor='black', alpha=0.8)
    ax.text(year + 0.5, i, f'{symbol} — {name}', va='center',
            fontsize=10, fontweight='bold')
    ax.text(year - 0.3, i, str(year), va='center', ha='right',
            fontsize=9, color='gray')
    ax.text(1890, i, f'by {discoverer}', va='center',
            fontsize=8, color='gray', style='italic')

ax.set_xlim(1858, 1900)
ax.set_yticks([])
ax.set_xlabel("Year / 연도", fontsize=13)
ax.set_title("Elements Discovered by Spectral Analysis (1860–1885)\n"
             "분광법으로 발견된 원소들 — Kirchhoff & Bunsen's Legacy",
             fontsize=14)

# Mark the K&B paper.
ax.axvline(1860, color='red', linestyle='--', linewidth=2, alpha=0.5)
ax.text(1860, len(discoveries) - 0.5,
        '← K&B 1860\n    Paper', fontsize=10, color='red',
        fontweight='bold')

# Highlight helium discovery.
ax.annotate('He found in Sun\n27 years before\nfound on Earth!',
            xy=(1868, 4), xytext=(1875, 6),
            fontsize=9, color='darkorange', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='darkorange'))

ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("\nKirchhoff & Bunsen의 분광 분석법은 25년간 9개 이상의 새로운 원소 발견을 가능하게 했습니다.")
print("가장 극적인 사례: 헬륨(He)은 지구에서 발견되기 27년 전에 태양 스펙트럼에서 먼저 발견되었습니다!")

## Summary / 요약

| Part | 구현 내용 | 핵심 원리 | 다음 논문과의 연결 |
|------|---------|---------|---------------|
| 1 | 6개 원소의 방출선 스펙트럼 시각화 | 각 원소 = 고유한 스펙트럼 지문 | #4 Schwarzschild: 흡수선 형성 조건의 정량적 모델 |
| 2 | 불꽃 온도 추정 ($t = \sum gw / \sum ps$) | 에너지 보존 법칙의 열역학적 응용 | #4: 태양 대기의 온도 구조 |
| 3 | 온도 불변성 시뮬레이션 | 선 위치 = 원자 고유 성질 (에너지 준위) | Bohr (1913): 양자역학적 설명 |
| 4 | 방출선 ↔ Fraunhofer 흡수선 대조 | 같은 원소 → 같은 파장의 방출/흡수 | #5 Hale: Zeeman 효과로 자기장 측정 |
| 5 | 검출 민감도 비교 | 분광법 >> 습식 화학 (10⁵배) | ICP-MS 등 현대 분석 기법으로 진화 |
| 6 | 혼합 시료 시간 분해 분석 | 휘발성 차이에 의한 순차적 검출 | 현대 크로마토그래피의 선구적 개념 |
| 7 | 스펙트럼 지문 식별 알고리즘 | 파장 매칭 = 원소 식별 | 현대 자동 스펙트럼 분석 소프트웨어 |
| 8 | 분광법 발견 원소 타임라인 | 분석 도구 → 발견 도구 | He(1868): 태양에서 먼저 발견! |